In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
import requests
from bs4 import BeautifulSoup

data = []

headers = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/115.0.0.0 Safari/537.36")
}

for page in range(2, 14):

    url = f'https://www.flipkart.com/search?q=laptop&page={page}'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")

    cards = soup.find_all("div", class_="ZFwe0M row")

    for i in cards:

        name_tag = i.find("div", class_="RG5Slk")
        Name = name_tag.get_text(strip=True) if name_tag else None

        actual_tag = i.find("div", class_="kRYCnD gxR4EY")
        Actual_Price = actual_tag.get_text(strip=True) if actual_tag else None

        disc_price_tag = i.find("div", class_="hZ3P6w DeU9vF")
        Discounted_Price = disc_price_tag.get_text(strip=True) if disc_price_tag else None

        disc_tag = i.find("div", class_="HQe8jr")
        Discount = disc_tag.get_text(strip=True) if disc_tag else 0

        rating_tag = i.find("div", class_="MKiFS6")
        Rating = rating_tag.get_text(strip=True) if rating_tag else 0

        data.append([Name, Actual_Price, Discounted_Price, Discount, Rating])

# Creating DataFrame

In [ ]:
import pandas as pd
df=pd.DataFrame(data,columns=['Name', 'Actual_Price', 'Discounted_Price', 'Discount%', 'Rating'])
df

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 288 entries, 0 to 287
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Name              288 non-null    object
 1   Actual_Price      284 non-null    object
 2   Discounted_Price  288 non-null    object
 3   Discount%         288 non-null    object
 4   Rating            288 non-null    object
dtypes: object(5)
memory usage: 11.4+ KB


In [ ]:
df.describe()

,Name,Actual_Price,Discounted_Price,Discount%,Rating
count,288,284,288,288,288
unique,158,123,102,50,16
top,HP Omnibook 3 AMD Ryzen AI 5 Hexa Core 340 - (...,"₹76,295","₹64,990",27% off,4.3
freq,13,13,16,29,74


In [ ]:
df.shape

(288, 5)

# Handling Null Values

In [ ]:
df.isnull().sum()

Name                0
Actual_Price        4
Discounted_Price    0
Discount%           0
Rating              0
dtype: int64

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.shape

(284, 5)

# Formatting & Changing DataTypes

In [ ]:
df['Actual_Price']=df['Actual_Price'].str.replace("₹","",regex=False)
df['Actual_Price']=df['Actual_Price'].str.replace(",","",regex=False)
df['Actual_Price']=pd.to_numeric(df['Actual_Price'])
df.dtypes

Name                object
Actual_Price         int64
Discounted_Price    object
Discount%           object
Rating              object
dtype: object

In [ ]:
df['Discounted_Price']=df['Discounted_Price'].str.replace("₹","",regex=False)
df['Discounted_Price']=df['Discounted_Price'].str.replace(",","",regex=False)
df['Discounted_Price']=pd.to_numeric(df['Discounted_Price'])
df.dtypes

Name                object
Actual_Price         int64
Discounted_Price     int64
Discount%           object
Rating              object
dtype: object

In [ ]:
df['Rating']=pd.to_numeric(df['Rating'])
df.dtypes

Name                 object
Actual_Price          int64
Discounted_Price      int64
Discount%            object
Rating              float64
dtype: object

In [ ]:
df['Discount%']= df['Discount%'].str.extract(r'(\d+)')
df['Discount%'] = pd.to_numeric(df['Discount%'])
df.dtypes


Name                 object
Actual_Price          int64
Discounted_Price      int64
Discount%           float64
Rating              float64
dtype: object

In [ ]:
df

,Name,Actual_Price,Discounted_Price,Discount%,Rating
0,Acer Aspire 3 Intel Core i3 13th Gen 1305U - (...,44999,39990,11.0,4.3
1,HP Victus AMD Ryzen 7 Octa Core 260 - (24 GB/1...,120928,102990,14.0,4.5
2,MOTOROLA Motobook 60 Full Metal OLED (i5 14th ...,93690,49990,46.0,4.4
3,MOTOROLA Motobook 60 Full Metal OLED (i7 14th ...,104890,57990,44.0,4.4
4,Lenovo Chromebook Duet 11M889 MediaTek MediaTe...,37690,19990,46.0,4.2
...,...,...,...,...,...
283,HP OmniBook 7 Aero (Previously Pavilion) AMD R...,109900,76851,30.0,3.9
284,MSI Modern 14 AMD Ryzen 5 Hexa Core 7430U - (1...,66990,42990,35.0,4.4
285,"MICROSOFT Surface Pro 12"" with Type Cover Core...",118198,112822,4.0,0.0
286,ASUS ExpertBook P1 (i7 14th Gen) Intel Core 7 ...,129990,83990,35.0,0.0


# Inserting New Column at Desired Location

In [ ]:
df.insert(4,"Discounted_Amount",df['Actual_Price']-df['Discounted_Price'])
df

,Name,Actual_Price,Discounted_Price,Discount%,Discounted_Amount,Rating
0,Acer Aspire 3 Intel Core i3 13th Gen 1305U - (...,44999,39990,11.0,5009,4.3
1,HP Victus AMD Ryzen 7 Octa Core 260 - (24 GB/1...,120928,102990,14.0,17938,4.5
2,MOTOROLA Motobook 60 Full Metal OLED (i5 14th ...,93690,49990,46.0,43700,4.4
3,MOTOROLA Motobook 60 Full Metal OLED (i7 14th ...,104890,57990,44.0,46900,4.4
4,Lenovo Chromebook Duet 11M889 MediaTek MediaTe...,37690,19990,46.0,17700,4.2
...,...,...,...,...,...,...
283,HP OmniBook 7 Aero (Previously Pavilion) AMD R...,109900,76851,30.0,33049,3.9
284,MSI Modern 14 AMD Ryzen 5 Hexa Core 7430U - (1...,66990,42990,35.0,24000,4.4
285,"MICROSOFT Surface Pro 12"" with Type Cover Core...",118198,112822,4.0,5376,0.0
286,ASUS ExpertBook P1 (i7 14th Gen) Intel Core 7 ...,129990,83990,35.0,46000,0.0


In [ ]:
df.dtypes

Name                  object
Actual_Price           int64
Discounted_Price       int64
Discount%            float64
Discounted_Amount      int64
Rating               float64
dtype: object

# Saving the Scrapped file to csv

In [ ]:
df.to_csv("Laptop.xlsx",index=False)

In [ ]:
file = pd.read_csv('Laptop')
file

,Name,Actual_Price,Discounted_Price,Discount%,Discounted_Amount,Rating
0,Acer Aspire 3 Intel Core i3 13th Gen 1305U - (...,44999,39990,11.0,5009,4.3
1,HP Victus AMD Ryzen 7 Octa Core 260 - (24 GB/1...,120928,102990,14.0,17938,4.5
2,MOTOROLA Motobook 60 Full Metal OLED (i5 14th ...,93690,49990,46.0,43700,4.4
3,MOTOROLA Motobook 60 Full Metal OLED (i7 14th ...,104890,57990,44.0,46900,4.4
4,Lenovo Chromebook Duet 11M889 MediaTek MediaTe...,37690,19990,46.0,17700,4.2
...,...,...,...,...,...,...
279,HP OmniBook 7 Aero (Previously Pavilion) AMD R...,109900,76851,30.0,33049,3.9
280,MSI Modern 14 AMD Ryzen 5 Hexa Core 7430U - (1...,66990,42990,35.0,24000,4.4
281,"MICROSOFT Surface Pro 12"" with Type Cover Core...",118198,112822,4.0,5376,0.0
282,ASUS ExpertBook P1 (i7 14th Gen) Intel Core 7 ...,129990,83990,35.0,46000,0.0
